In [ ]:
# ============================================================
# SalesInsight PY - Analise de Dados de Vendas
# RF01 - Geracao do dataset sintetico de vendas
# ============================================================
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta


def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # sujeira proposital para a etapa de limpeza
        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())

Dataset gerado com 200 registros.
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computadores       Sul   
4         5     2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   

   quantidade  preco_unitario  
0           2          102.90  
1           9          105.13  
2           9         2267.33  
3           4         1181.72  
4           7          114.25  


In [4]:
# ============================================================
# RF02 - Inspecao inicial do dataset
# ============================================================
def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df


inspecionar_dados(df_bruto)


=== INSPECAO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Com

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
...,...,...,...,...,...,...,...,...
195,196,2025-04-21,Cliente_028,Headset,Perifericos,Sul,9,373.19
196,197,2025-07-24,Cliente_027,Monitor,Computadores,Nordeste,6,1317.87
197,198,2025-07-11,Cliente_010,Monitor,Computadores,Centro-Oeste,6,1259.44
198,199,2025-02-17,Cliente_028,Notebook,Computadores,Nordeste,2,3848.36


In [ ]:
import os
print("Arquivos na pasta:", os.listdir())

Arquivos na pasta: ['.config', 'vendas.csv', 'sample_data']


In [5]:
# ============================================================
# RF03 - Tratamento dos dados (limpeza)
# ============================================================
def tratar_dados(df):
    n_inicial = len(df)

    colunas_texto = ["data_venda", "cliente", "produto", "categoria", "regiao"]
    for col in colunas_texto:
        df[col] = df[col].astype(str).str.strip()

    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    n_datas_invalidas = df["data_venda"].isna().sum()
    df = df.dropna(subset=["data_venda"]).copy()

    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    n_final = len(df)

    relatorio = {
        "registros_iniciais": n_inicial,
        "datas_invalidas_removidas": int(n_datas_invalidas),
        "registros_finais": n_final,
        "registros_removidos_total": n_inicial - n_final,
    }

    print("\n=== RELATORIO DE LIMPEZA ===")
    print(f"Registros iniciais:          {relatorio['registros_iniciais']}")
    print(f"Datas invalidas removidas:   {relatorio['datas_invalidas_removidas']}")
    print(f"Registros finais:            {relatorio['registros_finais']}")
    print(f"Total removido:              {relatorio['registros_removidos_total']}")

    return df, relatorio


df_limpo, relatorio_limpeza = tratar_dados(df_bruto.copy())
print("\nTipos apos limpeza:")
print(df_limpo.dtypes)


=== RELATORIO DE LIMPEZA ===
Registros iniciais:          200
Datas invalidas removidas:   6
Registros finais:            194
Total removido:              6

Tipos apos limpeza:
id_venda                   int64
data_venda        datetime64[ns]
cliente                   object
produto                   object
categoria                 object
regiao                    object
quantidade                 int64
preco_unitario           float64
dtype: object


In [6]:
# ============================================================
# RF04 - Colunas derivadas com transformacoes condicionais
# ============================================================
def criar_colunas_derivadas(df):
    """Cria receita_total, mes, mes_nome, trimestre, ano e faixa_receita_item."""

    # 1. receita por linha (vetorizado)
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # 2. componentes da data
    df["mes"] = df["data_venda"].dt.month
    df["ano"] = df["data_venda"].dt.year
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)

    # 3. nome do mes em portugues (dicionario)
    mapa_meses = {
        1: "Janeiro", 2: "Fevereiro", 3: "Marco",    4: "Abril",
        5: "Maio",    6: "Junho",     7: "Julho",    8: "Agosto",
        9: "Setembro",10: "Outubro",  11: "Novembro",12: "Dezembro",
    }
    df["mes_nome"] = df["mes"].map(mapa_meses)

    # 4. faixa de receita do item (condicional vetorizada com np.select)
    condicoes = [
        df["receita_total"] < 500,
        (df["receita_total"] >= 500) & (df["receita_total"] < 5000),
        df["receita_total"] >= 5000,
    ]
    faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
    df["faixa_receita_item"] = np.select(condicoes, faixas, default="Nao Classificado")

    return df


# --- executando ---
df_limpo = criar_colunas_derivadas(df_limpo)

print("\n=== COLUNAS DERIVADAS CRIADAS ===")
print(df_limpo[["data_venda", "receita_total", "mes", "mes_nome",
                "trimestre", "ano", "faixa_receita_item"]].head(10))

print("\nDistribuicao por faixa de receita:")
print(df_limpo["faixa_receita_item"].value_counts())

print("\nDistribuicao por trimestre:")
print(df_limpo["trimestre"].value_counts().sort_index())


=== COLUNAS DERIVADAS CRIADAS ===
   data_venda  receita_total  mes  mes_nome trimestre   ano faixa_receita_item
0  2025-05-21         205.80    5      Maio        Q2  2025        Baixo Valor
2  2025-10-15       20405.97   10   Outubro        Q4  2025         Alto Valor
3  2025-05-23        4726.88    5      Maio        Q2  2025        Medio Valor
4  2025-03-21         799.75    3     Marco        Q1  2025        Medio Valor
5  2025-07-03        6747.84    7     Julho        Q3  2025         Alto Valor
6  2025-07-13         970.40    7     Julho        Q3  2025        Medio Valor
7  2025-12-27        2148.72   12  Dezembro        Q4  2025        Medio Valor
8  2025-04-30        2004.75    4     Abril        Q2  2025        Medio Valor
9  2025-07-01         647.16    7     Julho        Q3  2025        Medio Valor
10 2025-05-06         757.68    5      Maio        Q2  2025        Medio Valor

Distribuicao por faixa de receita:
faixa_receita_item
Alto Valor     87
Medio Valor    81
Baixo